
# Simulación epidemiológica básica con Python y Google Colab

## Notebook introductorio para clase

Este notebook está pensado como una **introducción muy básica** a los estudios de simulación.  
No presupone conocimientos previos de programación ni de modelización.

### Qué pretende enseñar este notebook
Al terminar este recorrido, el estudiantado debería poder:

- entender qué es una simulación y para qué sirve
- reconocer la lógica básica de un modelo epidemiológico sencillo
- identificar las partes principales de un modelo
- ejecutar escenarios distintos cambiando algunos parámetros
- interpretar con cautela los resultados

### Idea central
Una simulación **no observa directamente la realidad**.  
Lo que hace es construir una versión simplificada de un fenómeno y explorar qué ocurre si cambian ciertas condiciones.

En este caso usaremos un modelo epidemiológico muy simple para representar la propagación de una enfermedad infecciosa en una población.



## 1. Qué es un estudio de simulación

Un estudio de simulación parte de una pregunta del tipo:

- ¿Qué podría pasar si...?
- ¿Cómo cambiaría el resultado si modificamos una intervención?
- ¿Qué efecto tendría reducir los contactos entre personas?
- ¿Qué ocurre si una parte de la población se protege mejor?

A diferencia de un estudio empírico, aquí no recogemos datos nuevos del mundo real para responder directamente a la pregunta.  
En cambio, construimos un **modelo** con supuestos explícitos y observamos cómo se comporta.

### Ejemplo de preguntas que sí encajan bien con simulación
- ¿Qué podría pasar si se reduce el contacto social?
- ¿Qué podría pasar si hay un mandato de mascarilla?
- ¿Qué podría pasar si la enfermedad es más contagiosa?

### Idea importante
Los resultados de una simulación **dependen de los supuestos del modelo**.  
Por eso no deben interpretarse como “predicciones seguras”, sino como una manera de pensar escenarios y relaciones entre variables.



## 2. El modelo que vamos a usar: SIR

Vamos a usar un modelo clásico y muy sencillo llamado **SIR**.

La población se divide en tres grupos:

- **S - Susceptibles**: personas que todavía pueden infectarse
- **I - Infectadas**: personas que tienen la infección y pueden contagiar
- **R - Recuperadas**: personas que ya no contagian y que salen de la dinámica del contagio

### Cómo funciona la lógica del modelo
En cada paso del tiempo:

- algunas personas susceptibles pasan a infectadas
- algunas personas infectadas pasan a recuperadas

### Qué simplifica este modelo
Este modelo es útil para aprender, pero simplifica mucho la realidad.  
Por ejemplo, no distingue entre edades, gravedad clínica, reinfección, vacunación o redes sociales de contacto.

Aun así, es muy útil para entender la **anatomía básica** de una simulación.



## 3. Las piezas principales del modelo

Antes de ejecutar nada, conviene identificar las partes del modelo.

### Estado del sistema
Es la situación de la población en cada momento:
- cuántas personas son susceptibles
- cuántas están infectadas
- cuántas se han recuperado

### Parámetros
Son valores que controlan el comportamiento del modelo:
- tamaño de la población
- contagiosidad
- velocidad de recuperación
- duración de la simulación

### Reglas
Son las instrucciones que dicen cómo cambia el sistema con el tiempo.

### Salidas
Son los resultados que observamos:
- número de infectados a lo largo del tiempo
- pico de la infección
- número final de personas recuperadas

### Intervenciones
Más adelante añadiremos dos intervenciones sencillas:
- **reducción de contactos** tipo confinamiento
- **reducción de transmisión** tipo mascarilla

Esto nos permitirá comparar escenarios.


In [ ]:

# 4. Librerías necesarias
# Esta celda importa las herramientas que vamos a usar.
# No hace falta modificar nada.

import numpy as np
import matplotlib.pyplot as plt



## 4. Función del modelo

La siguiente celda contiene la función principal de la simulación.

No hace falta entender cada línea de código para seguir el notebook.  
Lo importante aquí es reconocer que el modelo necesita:

- valores iniciales
- parámetros
- reglas de cambio
- una forma de guardar resultados

Más abajo usaremos esta función para ejecutar distintos escenarios.


In [ ]:

def simulate_sir(
    population=10000,
    initial_infected=10,
    beta=0.30,
    gamma=0.10,
    days=120,
    lockdown_start=None,
    lockdown_reduction=0.0,
    mask_start=None,
    mask_effect=0.0
):
    """
    Modelo SIR discreto y simple.

    Parámetros:
    - population: tamaño de la población
    - initial_infected: personas infectadas al inicio
    - beta: tasa base de transmisión
    - gamma: tasa de recuperación diaria
    - days: duración de la simulación
    - lockdown_start: día en que empieza una reducción de contactos
    - lockdown_reduction: proporción de reducción del contacto (0 a 1)
    - mask_start: día en que empieza la medida tipo mascarilla
    - mask_effect: proporción de reducción adicional de la transmisión (0 a 1)
    """

    S = np.zeros(days + 1)
    I = np.zeros(days + 1)
    R = np.zeros(days + 1)

    S[0] = population - initial_infected
    I[0] = initial_infected
    R[0] = 0

    for t in range(days):
        effective_beta = beta

        if lockdown_start is not None and t >= lockdown_start:
            effective_beta *= (1 - lockdown_reduction)

        if mask_start is not None and t >= mask_start:
            effective_beta *= (1 - mask_effect)

        new_infections = effective_beta * S[t] * I[t] / population
        new_recoveries = gamma * I[t]

        new_infections = min(new_infections, S[t])
        new_recoveries = min(new_recoveries, I[t])

        S[t + 1] = S[t] - new_infections
        I[t + 1] = I[t] + new_infections - new_recoveries
        R[t + 1] = R[t] + new_recoveries

    return {
        "days": np.arange(days + 1),
        "S": S,
        "I": I,
        "R": R
    }



## 5. Primer escenario: situación base

Empezamos con un escenario base sin ninguna intervención.

### Qué conviene observar
- cómo crece el número de personas infectadas
- cuándo aparece el pico
- qué ocurre después con la recuperación

La salida del modelo es un conjunto de curvas.


In [ ]:

baseline = simulate_sir(
    population=10000,
    initial_infected=10,
    beta=0.30,
    gamma=0.10,
    days=120
)

plt.figure(figsize=(10, 5))
plt.plot(baseline["days"], baseline["S"], label="Susceptibles")
plt.plot(baseline["days"], baseline["I"], label="Infectadas")
plt.plot(baseline["days"], baseline["R"], label="Recuperadas")
plt.xlabel("Días")
plt.ylabel("Número de personas")
plt.title("Escenario base del modelo SIR")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()



## 6. Preguntas de interpretación

Responde brevemente antes de seguir.

1. ¿En qué momento parece alcanzarse el pico de personas infectadas?
2. ¿Qué curva crece al principio más rápido?
3. ¿Qué ocurre con el número de personas susceptibles a lo largo del tiempo?
4. ¿Qué nos dice esta figura sobre la lógica de propagación del modelo?

### Idea didáctica
Aquí no interesa “acertar” una respuesta perfecta.  
Interesa empezar a leer un modelo como una representación dinámica de un fenómeno.



## 7. Escenario con reducción de contactos

Ahora vamos a introducir una medida simple tipo **confinamiento o reducción de contactos**.

### Qué cambia en el modelo
No cambiamos la población ni la enfermedad.  
Solo reducimos el número efectivo de contactos desde un día concreto.

Esto se representa reduciendo la tasa de transmisión a partir del día indicado.


In [ ]:

lockdown = simulate_sir(
    population=10000,
    initial_infected=10,
    beta=0.30,
    gamma=0.10,
    days=120,
    lockdown_start=20,
    lockdown_reduction=0.45
)

plt.figure(figsize=(10, 5))
plt.plot(baseline["days"], baseline["I"], label="Infectadas - base")
plt.plot(lockdown["days"], lockdown["I"], label="Infectadas - con reducción de contactos")
plt.xlabel("Días")
plt.ylabel("Número de personas infectadas")
plt.title("Comparación del escenario base y un escenario con reducción de contactos")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()



## 8. Escenario con mascarilla

Añadimos ahora una segunda intervención sencilla, que reduce la transmisión desde un cierto día.

### Cómo interpretarla
No estamos modelando mascarillas con realismo completo.  
Solo usamos una reducción de la transmisión para representar una intervención que hace menos probable el contagio.


In [ ]:

mask = simulate_sir(
    population=10000,
    initial_infected=10,
    beta=0.30,
    gamma=0.10,
    days=120,
    mask_start=20,
    mask_effect=0.30
)

plt.figure(figsize=(10, 5))
plt.plot(baseline["days"], baseline["I"], label="Infectadas - base")
plt.plot(mask["days"], mask["I"], label="Infectadas - con mascarilla")
plt.xlabel("Días")
plt.ylabel("Número de personas infectadas")
plt.title("Comparación del escenario base y un escenario con medida tipo mascarilla")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()



## 9. Escenario combinado

Ahora combinamos las dos intervenciones.

### Idea para la discusión
En simulación es frecuente comparar escenarios porque muchas preguntas son comparativas:

- ¿qué cambia?
- ¿cuánto cambia?
- ¿cuándo cambia?
- ¿qué intervención parece más influyente en este modelo?


In [ ]:

combined = simulate_sir(
    population=10000,
    initial_infected=10,
    beta=0.30,
    gamma=0.10,
    days=120,
    lockdown_start=20,
    lockdown_reduction=0.45,
    mask_start=20,
    mask_effect=0.30
)

plt.figure(figsize=(10, 5))
plt.plot(baseline["days"], baseline["I"], label="Base")
plt.plot(lockdown["days"], lockdown["I"], label="Reducción de contactos")
plt.plot(mask["days"], mask["I"], label="Mascarilla")
plt.plot(combined["days"], combined["I"], label="Combinado")
plt.xlabel("Días")
plt.ylabel("Número de personas infectadas")
plt.title("Comparación de varios escenarios")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()



## 10. Una función pequeña para resumir resultados

A veces un gráfico no basta.  
También conviene resumir algunos resultados clave, por ejemplo:

- día del pico
- tamaño del pico
- número final de personas recuperadas

Esto ayuda a comparar escenarios de forma más clara.


In [ ]:

def summarise_results(simulation):
    peak_day = int(np.argmax(simulation["I"]))
    peak_infected = float(np.max(simulation["I"]))
    final_recovered = float(simulation["R"][-1])
    return {
        "día del pico": peak_day,
        "pico de infectadas": round(peak_infected, 1),
        "recuperadas al final": round(final_recovered, 1)
    }

summary = {
    "Base": summarise_results(baseline),
    "Reducción de contactos": summarise_results(lockdown),
    "Mascarilla": summarise_results(mask),
    "Combinado": summarise_results(combined)
}

summary



## 11. Qué estamos aprendiendo realmente

Este notebook no pretende enseñar epidemiología avanzada ni programación completa.

Pretende enseñar algo más básico y muy importante:

### La anatomía de un estudio de simulación
- hay una pregunta
- hay supuestos
- hay parámetros
- hay reglas
- hay escenarios
- hay resultados
- hay interpretación

### Lo más importante
Los resultados no “salen de la nada”.  
Salen de la estructura del modelo y de los valores elegidos.

Por eso, interpretar una simulación exige preguntar siempre:

- ¿qué se ha simplificado?
- ¿qué no está representado?
- ¿qué significan exactamente estos parámetros?
- ¿qué conclusión sería razonable y cuál sería exagerada?



## 12. Preguntas finales para el tutorial

Puedes usar estas preguntas como cierre o discusión.

1. ¿Qué diferencia principal ves entre este estudio de simulación y un estudio empírico?
2. ¿Qué parte del modelo te parece más importante para entender los resultados?
3. ¿Qué limitaciones tiene este modelo para representar una epidemia real?
4. ¿Qué otra intervención sencilla se podría añadir al modelo?
5. ¿Qué preguntas podría ayudar a responder este modelo y cuáles no?

## 13. Posibles ampliaciones más adelante
Si quisieras hacer el modelo un poco más realista en el futuro, podrías añadir:

- vacunación
- grupos de edad
- distintas tasas de contacto
- hospitalización
- mortalidad
- redes de contacto en lugar de mezcla homogénea

Pero para una primera aproximación, este nivel de simplicidad es útil y suficiente.
